# Geomorphometry in GRASS - Ponui Island

This notebook accompanies **Chapter 12 (Geomorphometry in GRASS)** of the *Geomorphometry* book. It reproduces the chapter's analyses on the Ponui
Island LiDAR data using GRASS 8.5+ and `grass.jupyter` for inline rendering.

Compared to `geomorphometry_in_grass.py`, this notebook removes abstraction so every GRASS command and every rendering step is visible. It is intended as a learning aid -- the `.py` script remains the canonical figure-producer for the book.

## Requirements

- GRASS **8.5+** available as `grass` on `PATH`.
- The GRASS add-ons listed in `gextensions.txt` (the first code cell will
  install them).
- Ponui Island data downloaded from [Zenodo](https://doi.org/10.5281/zenodo.18314107) and placed in `CH12/data/`:
  - `data/DEM_ponui_island_dsm.tif`
  - `data/DEM_ponui_island_dtm.tif`
  - `data/LAS_ponui_island_lidar.laz`
- Python packages: `matplotlib`, `Pillow`, `numpy` (see `requirements.txt`).

## 1. Setup

Configure paths and add the GRASS Python libraries to `sys.path` before importing them. `grass --config python_path` returns the directory that contains the `grass` Python package shipped with the installed GRASS binary.

In [ ]:
from io import StringIO
from pathlib import Path
import subprocess
import os
import shutil
import sys



PROJECT_DIR = Path.cwd()           # assumes the notebook lives in CH12/
PROJECT_NAME = PROJECT_DIR / "ponui"
MAPSET_NAME = "PERMANENT"
EPSG_CODE = "2193"                 # NZGD2000 / New Zealand Transverse Mercator

DSM_PATH = PROJECT_DIR / "data" / "DEM_ponui_island_dsm.tif"
DTM_PATH = PROJECT_DIR / "data" / "DEM_ponui_island_dtm.tif"
LIDAR_PATH = PROJECT_DIR / "data" / "LAS_ponui_island_lidar.laz"

# Raster names used throughout the workflow
DSM_NAME = "dsm_10m"
DTM_NAME = "dem_10m"
DTM_RELIEF = "dtm_relief"
LIDAR_DTM_10M = "lidar_dtm_10m"
LIDAR_DTM_1M = "lidar_dtm_1m"
LIDAR_DTM_1M_RELIEF = "lidar_dtm_1m_relief"
LIDAR_DTM_1M_SKYVIEW = "lidar_dtm_1m_skyview"

ISLAND_RESOLUTION = 10  # m
AOI_REGION = "aoi"
AOI_RESOLUTION = 1      # m

SAVE_DIR = PROJECT_DIR / "figures"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

COLOR_OCEAN = "#0F78BE"

# Define custom number of core for processing 
# Must be an integer > 0
NPROCS = None

def get_nprocs_workers(nproc = None) -> int:
    if nproc is None:
        if hasattr(os, "process_cpu_count"):
            return os.process_cpu_count()
    
        if hasattr(os, "sched_getaffinity"):
            try:
                return len(os.sched_getaffinity(0))
            except NotImplementedError:
                pass
        
        cores = os.cpu_count() or 1
        return min(32, cores + 4)

    if not isinstance(nproc, int) or isinstance(nproc, bool):
        raise TypeError(
            f"nproc must be an integer or None, got {type(nproc).__name__}"
        )

    if nproc < 1:
        raise ValueError(f"nproc must be 1 or greater, got {nproc}")

    return nproc


NPROCS = get_nprocs_workers()

# Make the GRASS Python libraries importable.
def _grass_python_path():
    """Locate the GRASS Python path across Linux, macOS, and Windows."""
    # If GISBASE is set, the Python path is <GISBASE>/etc/python; this skips
    # the launcher (handy on Windows).
    gisbase = os.environ.get("GISBASE")
    if gisbase and os.path.isdir(os.path.join(gisbase, "etc", "python")):
        return os.path.join(gisbase, "etc", "python")
    # shutil.which honors PATHEXT on Windows, so it resolves grass.bat;
    # GRASS_BIN overrides.
    launcher = os.environ.get("GRASS_BIN") or shutil.which("grass")
    if not launcher:
        raise RuntimeError(
            "GRASS launcher not found. Run inside a GRASS-enabled shell "
            "(the OSGeo4W Shell on Windows), or set GRASS_BIN or GISBASE."
        )
    # Windows batch launchers (grass.bat) must run through cmd.exe.
    cmd = [launcher, "--config", "python_path"]
    if os.name == "nt" and launcher.lower().endswith((".bat", ".cmd")):
        cmd = ["cmd", "/c", *cmd]
    return subprocess.check_output(cmd, text=True).strip()


grass_python_path = _grass_python_path()
if grass_python_path and grass_python_path not in sys.path:
    sys.path.append(grass_python_path)

import grass.script as gs
import grass.jupyter as gj
from grass.tools import Tools
from grass.exceptions import ScriptError

print(f"Python: {sys.version.split()[0]}")

### Create the GRASS project and start a session

We create a fresh GRASS project at `CH12/ponui/` (EPSG:2193), if it already exists, we re-use it. `gs.setup.init(...)` returns a session that we pass into `Tools(...)` to get an object whose attributes correspond to GRASS modules (`tools.r_import(...)` instead of `gs.run_command('r.import', ...)`).

In [ ]:
try:
    gs.create_project(path=PROJECT_NAME, epsg=EPSG_CODE)
except ScriptError:
    print(f"Project {PROJECT_NAME} already exists -- using it.")

session = gs.setup.init(PROJECT_NAME)
tools = Tools(session=session, overwrite=True)
tools.g_gisenv()

### Install GRASS add-ons

The workflow uses several add-ons (listed in `gextensions.txt`).
If they are not already installed in this GRASS environment, install them now.
Skip any that fail (e.g., if your installation already provides them).

In [ ]:
extensions = [
    "d.region.grid",
    "r.flowaccumulation",
    "r.stream.order",
    "r.stream.distance",
    "r.hand",
    "r.tpi",
    "r.skyview",
]

for ext in extensions:
    try:
        tools.g_extension(extension=ext)
        print(f"  installed: {ext}")
    except Exception as exc:
        print(f"  skipped {ext}: {exc}")

## 2. Import the base rasters and the 10 m LiDAR mean DTM

Three inputs are imported:

- `dsm_10m` -- the 10 m Digital Surface Model (top of canopy, buildings).
- `dem_10m` -- the 10 m Digital Terrain Model (bare earth).
- `lidar_dtm_10m` -- a 10 m mean elevation raster built directly from the
  raw LiDAR class-2 (ground) returns via `r.in.pdal`.

After import, we set ocean cells (elevation `<= 0`) to `NULL` so they do
not contaminate the analyses.

In [ ]:
tools.r_import(
    input=str(DSM_PATH), output=DSM_NAME,
    resample="bilinear", resolution="value",
    resolution_value=ISLAND_RESOLUTION,
    title=f"Ponui Island {ISLAND_RESOLUTION}m DSM",
)

tools.r_import(
    input=str(DTM_PATH), output=DTM_NAME,
    resample="bilinear", resolution="value",
    resolution_value=ISLAND_RESOLUTION,
    title=f"Ponui Island {ISLAND_RESOLUTION}m DTM",
)

tools.r_in_pdal(
    input=str(LIDAR_PATH), output=LIDAR_DTM_10M,
    method="mean", resolution=ISLAND_RESOLUTION,
    class_filter=2, flags="we",
)

In [ ]:
for raster in (DTM_NAME, DSM_NAME, LIDAR_DTM_10M):
    tools.r_null(map=raster, setnull="-9999-0")

tools.g_region(raster=DTM_NAME, flags="a")

## 3. Color schemes for the chapter palette

Color tables are applied with `r.colors`. In the `.py` script these live in a `GeoColors` class; here we inline each rule string so you can see and tweak the palette beside its use. We also paint an `ocean` background raster (a constant value over the full region) that we draw underneath the elevation rasters in every map.

In [ ]:
# Ponui Island elevation palette (greens to dark teal).
ELEVATION_PONUI_RULES = (
    "0%   #f3e7d3\n"
    "10%  #d8cfac\n"
    "20%  #b4c88a\n"
    "35%  #82b66a\n"
    "50%  #4f9d53\n"
    "65%  #2e7a42\n"
    "80%  #1f5631\n"
    "90%  #17483b\n"
    "100% #0d3a3a\n"
)

def apply_elevation_colors(map_names):
    if isinstance(map_names, str):
        map_names = [map_names]
    for m in map_names:
        tools.r_colors(map=m, rules=StringIO(ELEVATION_PONUI_RULES))

apply_elevation_colors([DTM_NAME, DSM_NAME, LIDAR_DTM_10M])

In [ ]:
# Solid-color ocean background, drawn underneath shaded elevation in every map.
tools.r_mapcalc(expression=f"ocean = 1")
tools.r_colors(map="ocean", rules=StringIO(f"1 {COLOR_OCEAN}\n"))

## 4. Island-scale relief, second-order derivatives, and resampled DEMs

`r.relief` produces a hillshade for cartography. `r.slope.aspect` produces the chapter's four core second-order derivatives,
- slope
- aspect
- profile curvature
- tangential curvature

plus the partial derivatives `dx`,`dy` that `r.sim.water` / `r.sim.sediment` will need later.

We also resample the DTM to 150 m / 250 m / 500 m to support multi-scale discussions in the chapter (these resamples don't drive figures here; they are saved for the reader to experiment with).

In [ ]:
tools.r_relief(input=DTM_NAME, output=DTM_RELIEF)

tools.r_slope_aspect(
    elevation=DTM_NAME,
    slope=f"{DTM_NAME}_slope",
    aspect=f"{DTM_NAME}_aspect",
    pcurvature=f"{DTM_NAME}_pcurv",
    tcurvature=f"{DTM_NAME}_tcurv",
    dx=f"{DTM_NAME}_dx",
    dy=f"{DTM_NAME}_dy",
)

for res in (150, 250, 500):
    with gs.RegionManager(res=res, raster=DTM_NAME):
        tools.r_resamp_interp(
            input=DTM_NAME,
            output=f"{DTM_NAME}_{res}m",
            method="bilinear",
        )

# Restore the native island resolution before drawing.
tools.g_region(raster=DTM_NAME, flags="a")

### 4a. Define the AOI boundary vector

Before the first figure we build the AOI vector so it can be overlaid on the full-island map. The AOI is a single 1 km x 1 km cell from a 1 km grid covering the region (chosen to highlight Oranga Bay).

In [ ]:
tools.v_mkgrid(map="grid_1k_1k", box="1000,1000")

### 4b. Full-island elevation figure

This is the canonical chapter figure for the island scale: ocean background, hillshaded elevation, AOI outline, scale bar, and a graticule with white labels. We compose the map step-by-step with `grass.jupyter.Map`.

In [ ]:
univar = tools.r_univar(map=DTM_NAME, format="json").json
out = SAVE_DIR / f"{DTM_NAME}.png"

m = gj.Map(width=800, use_region=True)
m.d_rast(map="ocean")
m.d_grid(
    size="00:01:10",
    color="#FDFDFD",
    fontsize=22,
    text_color="#FDFDFD",
    border_color="#FFFFFF",
    flags="ga",
)
m.d_shade(color=DTM_NAME, shade=DTM_RELIEF, flags="n")
m.d_vect(map=AOI_REGION, type="boundary", color="#E66101", width=3)
m.d_text(
    text="Oranga Bay", at=(34, 53), size=2.25,
    color="#FDFDFD", font="sans",
)
m.d_legend(
    raster=DTM_NAME, at="4,21,82,85",
    font="sans", fontsize=24,
    border_color="#FDFDFD", bgcolor=COLOR_OCEAN, color="#FDFDFD",
    units="m", range=f"{univar['min']},{univar['max']}",
    flags="bt",
)
m.d_barscale(
    at=(1, 4), bgcolor="none", style="line", length=2, units="kilometers",
    color="#FDFDFD", font="sans", fontsize=24,
    flags="n",
)
m.save(filename=str(out))
m.show()

## 5. A small reusable AOI map helper

The remaining figures are nearly all AOI-scale maps with the same composition (ocean -> shaded raster -> optional overlay vectors/rasters -> graticule -> legend -> scale bar).
Rather than retype that 20+ times, we define **one small inline helper** here. Its body is short and directly visible, so you still see every GRASS / `gj.Map` call -- it's just parameterized over the raster, the legend, and any overlays. This is the only abstraction in the notebook. Compare with the much larger `aoi_map_figure(...)` in the `.py` script.

In [ ]:
def aoi_map(
    map_name,
    relief=LIDAR_DTM_1M_RELIEF,
    *,
    legend=None,
    legend_title="",
    legend_units="",
    legend_flags="t",
    legend_at="5,10,20,80",
    legend_range_min=None,
    legend_range_max=None,
    shade_flags="n",
    save_name=None,
    extra_rasters=None,
    extra_vectors=None,
    extra_shades=None,
):
    """Render an AOI-scale map and save PNG to figures/."""
    save_name = save_name or map_name
    out = SAVE_DIR / f"{save_name}_aoi.png"

    univar = tools.r_univar(map=map_name, format="json").json
    rmin = legend_range_min if legend_range_min is not None else univar["min"]
    rmax = legend_range_max if legend_range_max is not None else univar["max"]

    with gs.RegionManager(
        region=AOI_REGION, s="s-200", raster=LIDAR_DTM_1M,
        res=AOI_RESOLUTION, flags="a",
    ):
        m = gj.Map(width=800, use_region=True)
        m.d_rast(map="ocean")
        for sh in (extra_shades or []):
            m.d_shade(**sh)
        m.d_shade(color=map_name, shade=relief, flags=shade_flags)
        for r in (extra_rasters or []):
            m.d_rast(**r)
        for v in (extra_vectors or []):
            m.d_vect(**v)
        m.d_grid(
            size="00:00:10", color="#FDFDFD", text_color="#FDFDFD",
            border_color="#FFFFFF", fontsize=16, flags="gac",
        )
        m.d_text(
            text="Oranga Bay", at=(20, 84), size=4,
            color="white", font="sans",
        )
        m.d_legend(
            raster=legend or map_name, at=legend_at,
            font="sans", fontsize=21,
            border_color="none",
            title=legend_title, title_fontsize=24,
            range=f"{rmin},{rmax}",
            flags=legend_flags,
        )
        m.d_barscale(
            at=(60, 22), font="sans", fontsize=21,
            length=250, bgcolor="none", style="line", color="#FDFDFD",
            flags="n",
        )
        m.save(filename=str(out))
        return m

## 6. Define the AOI region and ingest LiDAR ground points

The chapter focuses on a 1 km x 1 km Area of Interest at Oranga Bay.
We set and save a region at 1 m resolution, then import the LiDAR ground returns (class 2) two ways:

- `v.in.pdal` -- as a vector point cloud (`lidar_be`), needed for the RST
  interpolation.
- `r.in.pdal method=n` -- a raster counting points per 1 m cell, useful
  for diagnosing the interpolation.

In [ ]:
tools.g_region(
    save=AOI_REGION,
    n=5918600.13, s=5917599.13,
    w=1793519.48, e=1794520.48,
    res=AOI_RESOLUTION,
    flags="ap",
)

In [ ]:
with gs.RegionManager(region=AOI_REGION, raster=LIDAR_DTM_1M if False else None,
                       res=AOI_RESOLUTION, flags="a"):
    tools.v_in_pdal(
        input=str(LIDAR_PATH), output="lidar_be",
        class_filter=2, flags="or",
    )
    tools.r_in_pdal(
        input=str(LIDAR_PATH), output="lidar_dtm_n_1m",
        method="n", resolution=AOI_RESOLUTION,
        class_filter=2, flags="we",
    )

### 6a. Interpolate the 1 m LiDAR DTM with Regularized Spline with Tension

`v.surf.rst` interpolates an elevation surface from the ground points and **simultaneously** computes its second-order derivatives (slope, aspect, profile tangential/mean curvature). Computing the derivatives directly from the spline fit gives smoother, less-aliased results than running `r.slope.aspect` on the interpolated raster afterwards.

Parameters here (`tension=300`, `smooth=0.1`, `npmin=200`, `dmin=1.5`) were chosen for the workflow.

In [ ]:
with gs.RegionManager(region=AOI_REGION, res=AOI_RESOLUTION, flags="a"):
    tools.v_surf_rst(
        input="lidar_be",
        elevation=LIDAR_DTM_1M,
        slope=f"{LIDAR_DTM_1M}_rst_slope",
        aspect=f"{LIDAR_DTM_1M}_rst_aspect",
        pcurvature=f"{LIDAR_DTM_1M}_rst_pcurv",
        tcurvature=f"{LIDAR_DTM_1M}_rst_tcurv",
        mcurvature=f"{LIDAR_DTM_1M}_rst_mcurv",
        tension=300, smooth=0.1, npmin=200, dmin=1.5, nprocs=NPROCS,
        flags="t",
    )
    tools.r_null(map=LIDAR_DTM_1M, setnull="-9999-0")
    tools.r_relief(input=LIDAR_DTM_1M, output=LIDAR_DTM_1M_RELIEF)
    apply_elevation_colors(LIDAR_DTM_1M)

### 6b. AOI elevation + bare-earth-points figures

Two diagnostic AOI maps: the interpolated 1 m DTM, and the per-cell point count from `r.in.pdal method=n`. The point-count map shows where the RST interpolation had a lot of constraints vs. where it had to interpolate across gaps.

In [ ]:
aoi_map(
    LIDAR_DTM_1M,
    legend_title="Elevation [m]",
    legend_units="m",
    legend_flags="st",
).show()

In [ ]:
tools.r_colors(map="lidar_dtm_n_1m", color="viridis", flags="e")
aoi_map(
    "lidar_dtm_n_1m",
    legend_title="Bare-earth points per cell",
    legend_flags="st",
).show()

## 7. AOI second-order derivatives + skyview

We run `r.slope.aspect` on the interpolated DTM to produce the standard second-order derivatives (note these are *different* from the RST derivatives produced by `v.surf.rst` -- `r.slope.aspect` works on the rasterized surface). We also compute `r.skyview` for nice cartography.

In [ ]:
with gs.RegionManager(region=AOI_REGION, res=AOI_RESOLUTION, flags="a"):
    tools.r_slope_aspect(
        elevation=LIDAR_DTM_1M,
        slope=f"{LIDAR_DTM_1M}_slope",
        aspect=f"{LIDAR_DTM_1M}_aspect",
        pcurvature=f"{LIDAR_DTM_1M}_pcurv",
        tcurvature=f"{LIDAR_DTM_1M}_tcurv",
        dx=f"{LIDAR_DTM_1M}_dx",
        dy=f"{LIDAR_DTM_1M}_dy",
    )
    try:
        tools.r_skyview(input=LIDAR_DTM_1M, output=LIDAR_DTM_1M_SKYVIEW, ndir=8)
    except Exception as exc:
        print(f"skipping r.skyview: {exc}")

### 7a. Color rules for slope, aspect, and curvature

We define the palettes for the derivatives. `aspect` uses a circular (colorblind-safe) ramp; `curvature` uses a symmetric ramp centered on 0 (positive = concave, negative = convex).

In [ ]:
ASPECT_CB_SAFE_RULES = (
    "0   #364B9A\n"
    "45  #4678C2\n"
    "90  #58A6D6\n"
    "135 #78C5BF\n"
    "180 #BED78D\n"
    "225 #E8C46B\n"
    "270 #E39EA7\n"
    "315 #AA7CC3\n"
    "360 #364B9A\n"
)

CURVATURE_RULES = (
    "-1.35 #08306B\n-1.0  #225EA8\n-0.7  #1D91C0\n-0.3  #41B6C4\n"
    "-0.1  #7FCDAB\n-0.01 #C7E9B4\n 0.0  #FFFFFF\n 0.01 #FFFFB2\n"
    " 0.1  #FEDA76\n 0.3  #FEB24C\n 0.7  #FD8D3C\n 1.0  #FC4E2A\n"
    " 1.35 #83006D\n"
)

for asp in (f"{LIDAR_DTM_1M}_aspect", f"{LIDAR_DTM_1M}_rst_aspect"):
    tools.r_colors(map=asp, rules=StringIO(ASPECT_CB_SAFE_RULES))

for curv in (
    f"{LIDAR_DTM_1M}_pcurv", f"{LIDAR_DTM_1M}_tcurv",
    f"{LIDAR_DTM_1M}_rst_pcurv", f"{LIDAR_DTM_1M}_rst_tcurv",
    f"{LIDAR_DTM_1M}_rst_mcurv",
):
    tools.r_colors(map=curv, rules=StringIO(CURVATURE_RULES))

tools.r_colors(map=f"{LIDAR_DTM_1M}_slope", color="sepia", flags="e")
tools.r_colors(map=f"{LIDAR_DTM_1M}_rst_slope", color="sepia", flags="e")

### 7b. Derivative figures (RST and raster-based)

Eight figures: slope / aspect / pcurv / tcurv from both the RST direct
derivatives and the post-hoc `r.slope.aspect` outputs. Compare them to
see how the spline-fit derivatives differ from rasterized ones.

In [ ]:
for kind in ("rst_", ""):
    aoi_map(f"{LIDAR_DTM_1M}_{kind}slope",
            legend_title="Slope [\u00b0]", legend_flags="st").show()
    aoi_map(f"{LIDAR_DTM_1M}_{kind}aspect",
            legend_title="Aspect [\u00b0]", legend_flags="st").show()
    aoi_map(f"{LIDAR_DTM_1M}_{kind}pcurv",
            legend_title="Profile curvature", legend_flags="t").show()
    aoi_map(f"{LIDAR_DTM_1M}_{kind}tcurv",
            legend_title="Tangential curvature", legend_flags="t").show()

aoi_map(f"{LIDAR_DTM_1M}_rst_mcurv",
        legend_title="Mean curvature", legend_flags="t").show()

## 8. Flow accumulation -- four methods compared

GRASS supports several flow-accumulation algorithms. We run four:

- **D8 MFD** (`r.watershed` with `-a4`): multiple flow direction, the
  default for hydrologic modelling.
- **D8 SFD** (`r.watershed` with `-sa`): single flow direction.
- **D-infinity SFD** (`r.flow`): Tarboton's continuous flow direction.
- **MEFA** (`r.flowaccumulation`, add-on): multiple-exclusive flow.

Each method produces a noticeably different drainage network on the same
DTM.

In [ ]:
with gs.RegionManager(region=AOI_REGION, raster=LIDAR_DTM_1M,
                       res=AOI_RESOLUTION, flags="a"):
    tools.r_watershed(
        elevation=LIDAR_DTM_1M,
        accumulation="d8_mfd_flowaccum",
        drainage="d8_mfd_flowdir",
        stream="d8_mfd_streams",
        basin="d8_mfd_basins2",
        threshold=100000, flags="a4",
    )
    tools.r_watershed(
        elevation=LIDAR_DTM_1M,
        accumulation="d8_sfd_flowaccum",
        drainage="d8_sfd_flowdir",
        threshold=100000, flags="sa",
    )
    tools.r_flow(elevation=LIDAR_DTM_1M,
                  flowaccumulation="dinf_sfd_flowaccum")
    tools.r_null(map="dinf_sfd_flowaccum", setnull="-9999-0")

    try:
        tools.r_flowaccumulation(
            input="d8_sfd_flowdir", format="45degree",
            output="MEFA_flowaccum", type="CELL",
        )
    except Exception as exc:
        print(f"skipping r.flowaccumulation: {exc}")

    # Log-stretch the accumulations so the figures are readable.
    for fa in ("d8_mfd_flowaccum", "d8_sfd_flowaccum",
               "dinf_sfd_flowaccum"):
        tools.r_colors(map=fa, color="water", flags="g")

### 8a. Vectorize the basins so they can overlay any AOI figure

In [ ]:
with gs.RegionManager(region=AOI_REGION, raster=LIDAR_DTM_1M,
                       res=AOI_RESOLUTION, flags="a"):
    tools.r_to_vect(input="d8_mfd_basins2", output="d8_mfd_basins2", type="area")
    tools.v_extract(input="d8_mfd_basins2", cats="2", output="basin2")

basin_overlay = [{"map": "d8_mfd_basins2", "type": "area",
                   "fill_color": "none", "color": "#F5F5F5", "width": 1}]
shade_under = [{"color": LIDAR_DTM_1M, "shade": LIDAR_DTM_1M_RELIEF, "flags": "n"}]

In [ ]:
for fa in ("d8_mfd_flowaccum", "d8_sfd_flowaccum",
           "dinf_sfd_flowaccum"):
    aoi_map(fa, legend_title="Flow accumulation", legend_flags="tl",
            legend_range_min=1,
            extra_vectors=basin_overlay, extra_shades=shade_under).show()

## 9. Topographic Wetness Index (TWI)

TWI = ln(a / tan(slope)), where `a` is the upslope contributing area per
unit contour length. We approximate `a` with the D8 MFD flow accumulation
and use the rasterized slope in radians.

High TWI = wet (convergent + flat), low TWI = dry (divergent + steep).

In [ ]:
TWI_RULES = (
    "0% #0571B0\n25% #92C5DE\n50% #F7F7F7\n"
    "75% #F4A582\n100% #CA0020\n"
)

with gs.RegionManager(region=AOI_REGION, raster=LIDAR_DTM_1M,
                       res=AOI_RESOLUTION, flags="a"):
    tools.r_mapcalc(expression=(
        f"twi = log(d8_mfd_flowaccum / "
        f"tan({LIDAR_DTM_1M}_slope * 3.14159 / 180))"
    ))
    tools.r_colors(map="twi", rules=StringIO(TWI_RULES))

aoi_map("twi", legend_title="TWI", legend_flags="t",
        extra_vectors=basin_overlay, extra_shades=shade_under).show()

## 10. Stream extraction and Horton / Strahler ordering

We extract streams from `r.watershed`'s stream raster (after thinning to a
single-cell skeleton) and then run `r.stream.extract` + `r.stream.order`
(add-ons) to assign Horton and Strahler stream orders.

In [ ]:
with gs.RegionManager(region=AOI_REGION, raster=LIDAR_DTM_1M,
                       res=AOI_RESOLUTION, flags="a"):
    tools.r_thin(input="d8_mfd_streams", output="d8_mfd_streams_thin")
    tools.r_to_vect(input="d8_mfd_streams_thin",
                     output="d8_mfd_streams", type="line")

    tools.r_stream_extract(
        elevation=LIDAR_DTM_1M, threshold=5000,
        direction="stream_extract_dir",
        stream_raster="stream_extract",
        stream_vector="stream_extract",
    )
    tools.r_stream_order(
        elevation=LIDAR_DTM_1M, accumulation="d8_mfd_flowaccum",
        direction="stream_extract_dir",
        stream_rast="stream_extract",
        stream_vect="stream_orders",
        strahler="strahler", horton="horton",
    )

In [ ]:
# Horton stream-order figure
aoi_map(
    LIDAR_DTM_1M, legend="horton",
    legend_title="Horton stream order", legend_flags="t",
    save_name="horton",
    extra_rasters=[{"map": "horton"}],
    extra_vectors=basin_overlay + [
        {"map": "stream_orders", "type": "line",
         "width_column": "horton", "width_scale": 2},
    ],
    extra_shades=shade_under,
).show()

# Strahler stream-order figure
aoi_map(
    LIDAR_DTM_1M, legend="strahler",
    legend_title="Strahler stream order", legend_flags="t",
    save_name="strahler",
    extra_rasters=[{"map": "strahler"}],
    extra_vectors=basin_overlay + [
        {"map": "stream_orders", "type": "line",
         "width_column": "strahler", "width_scale": 2},
    ],
    extra_shades=shade_under,
).show()

## 11. Height Above Nearest Drainage (HAND) + inundation

`r.hand` (add-on) computes HAND and, with `-t`, a space-time raster dataset (STRDS) of inundation extents at a range of water levels.

In [ ]:
with gs.RegionManager(region=AOI_REGION, raster=LIDAR_DTM_1M,
                       res=AOI_RESOLUTION, flags="a"):
    tools.r_hand(
        elevation=LIDAR_DTM_1M, threshold=50000,
        inundation_raster="inundation",
        inundation_strds="inundation_strds",
        start_water_level=0, end_water_level=5, water_level_step=0.5,
        hand="hand", flags="t",
    )

aoi_map("hand", legend_title="HAND [m]", legend_flags="bdt",
        extra_vectors=basin_overlay, extra_shades=shade_under).show()

## 12. Topographic Position Index (TPI)

`r.tpi` (add-on) computes the elevation difference between each cell and its neighborhood mean. Positive = ridges, negative = valleys, near-zero = flat/uniform slopes.

In [ ]:
TPI_RULES = (
    "0%   #0A4C6B\n25%  #DCF5FF\n50%  #FFF7DC\n"
    "75%  #FFE6DC\n100% #6B4E4C\n"
)

with gs.RegionManager(region=AOI_REGION, raster=LIDAR_DTM_1M,
                       res=AOI_RESOLUTION, flags="a"):
    tools.r_tpi(input=LIDAR_DTM_1M, output="tpi")
    tools.r_colors(map="tpi", rules=StringIO(TPI_RULES))

aoi_map("tpi", legend_title="TPI", legend_flags="t",
        extra_vectors=basin_overlay, extra_shades=shade_under).show()

## 13. Landform classification: geomorphons + multi-scale morphometry

Two complementary classifiers:

- `r.geomorphon` -- 10-class pattern-based landform classification.
- `r.param.scale` -- six-class morphometric classification at a chosen
  window size.

In [ ]:
with gs.RegionManager(region=AOI_REGION, raster=LIDAR_DTM_1M,
                       res=AOI_RESOLUTION, flags="a"):
    tools.r_geomorphon(
        elevation=LIDAR_DTM_1M,
        forms=f"{LIDAR_DTM_1M}_landforms",
        search=21, skip=1, flat=1, dist=0,
    )
    tools.r_param_scale(
        input=LIDAR_DTM_1M,
        output=f"{LIDAR_DTM_1M}_morphology",
        method="feature", size=5,
    )

aoi_map(f"{LIDAR_DTM_1M}_landforms",
        legend_title="Geomorphons", legend_flags="c",
        extra_shades=shade_under).show()
aoi_map(f"{LIDAR_DTM_1M}_morphology",
        legend_title="Morphometric features", legend_flags="c",
        extra_shades=shade_under).show()

## 14. Solar radiation

`r.sun` computes global radiation and insolation duration for given days of the year. We run it for the southern-hemisphere summer solstice (day 356) and winter solstice (day 172).

In [ ]:
SOLAR_RULES = (
    "0    #0b0b0b\n500  #0d1f3a\n1000 #1e4271\n1500 #3a78ab\n"
    "2000 #5aa5c9\n2500 #86c8dd\n3000 #b5e3f3\n4000 #f3f9d0\n"
    "5000 #fff3a0\n6000 #fed675\n7000 #fdb157\n8000 #f9833a\n"
    "8500 #e7552e\n9000 #cc301d\n9500 #a81c14\n10000 #7a0b0b\n"
)

with gs.RegionManager(region=AOI_REGION, raster=LIDAR_DTM_1M,
                       res=AOI_RESOLUTION, flags="a"):
    for day, tag in ((356, "summer"), (172, "winter")):
        tools.r_sun(
            elevation=LIDAR_DTM_1M,
            slope=f"{LIDAR_DTM_1M}_slope",
            aspect=f"{LIDAR_DTM_1M}_aspect",
            glob_rad=f"global_rad_{day}",
            insol_time=f"insol_time_{day}",
            day=day,
        )
        tools.r_colors(map=f"global_rad_{day}", rules=StringIO(SOLAR_RULES))
        aoi_map(f"global_rad_{day}",
                legend_title=f"Global radiation day {day} ({tag}) [Wh/m\u00b2]",
                legend_flags="t",
                extra_shades=shade_under).show()

## 15. Overland flow and erosion / deposition simulation

We mask the analysis region to a single drainage basin (basin 2 from `r.watershed`) and run path-sampling overland-flow (`r.sim.water`) and erosion-deposition (`r.sim.sediment`) simulations on it. Both modules use random walkers; the seed is fixed so the figures are reproducible.

In [ ]:
with gs.RegionManager(region=AOI_REGION, raster=LIDAR_DTM_1M,
                       res=AOI_RESOLUTION, flags="a"), \
     gs.MaskManager():
    tools.r_mask(vector="d8_mfd_basins2")

    tools.r_sim_water(
        elevation=LIDAR_DTM_1M,
        dx=f"{LIDAR_DTM_1M}_dx", dy=f"{LIDAR_DTM_1M}_dy",
        rain_value=30, infil_value=0.0, man_value=0.2,
        niterations=30, output_step=2,
        depth="depth", discharge="disch",
        random_seed=3, nwalkers=100000, nprocs=NPROCS,
        flags="t",
    )
    tools.r_mapcalc(
        expression="max_depth = if(depth.30 >= 0.01, depth.30, null())"
    )

    for expr in ("tranin = 0.001", "detin = 0.001", "shear_stress = 0.5"):
        tools.r_mapcalc(expression=expr)

    tools.r_sim_sediment(
        elevation=LIDAR_DTM_1M,
        dx=f"{LIDAR_DTM_1M}_dx", dy=f"{LIDAR_DTM_1M}_dy",
        water_depth="depth.30",
        detachment_coeff="detin", transport_coeff="tranin",
        shear_stress="shear_stress", man_value=0.04,
        transport_capacity="transport_capacity",
        tlimit_erosion_deposition="tlimit_erosion_deposition",
        sediment_concentration="sediment_concentration",
        sediment_flux="sediment_flux",
        erosion_deposition="erosion_deposition",
        niterations=30, output_step=2,
        random_seed=3, nprocs=NPROCS, nwalkers=100000,
    )

### 16. Render Maps

In [ ]:
for raster, title in [
    ("depth.30",                   "Water depth at t=30 [m]"),
    ("max_depth",                  "Max water depth [m]"),
    ("erosion_deposition",         "Erosion / deposition [kg/m\u00b2s]"),
    ("transport_capacity",         "Transport capacity [kg/m\u00b2s]"),
    ("tlimit_erosion_deposition",  "Transport-limited erosion/deposition [kg/m\u00b2s]"),
    ("sediment_flux",              "Sediment flux [kg/m\u00b2s]"),
    ("sediment_concentration",     "Sediment concentration [particle/m\u00b3]"),
]:
    aoi_map(raster, legend_title=title, legend_flags="t",
            extra_vectors=basin_overlay, extra_shades=shade_under).show()

## 16. 3D visualizations

`grass.jupyter.Map3D` wraps `m.nviz.image` to render NVIZ scenes inline. We render the AOI DTM draped with each of the key analysis rasters.

In [ ]:
def aoi_3d(mapcolor, elevation=LIDAR_DTM_1M, *, save_name=None,
            legend_units="", legend_flags="bt", legend_at="8,12,5,40",
            legend_range_min=None, legend_range_max=None):
    save_name = save_name or mapcolor
    out = SAVE_DIR / f"{save_name}_3d.png"

    univar = tools.r_univar(map=mapcolor, format="json").json
    rmin = legend_range_min if legend_range_min is not None else univar["min"]
    rmax = legend_range_max if legend_range_max is not None else univar["max"]

    m3 = gj.Map3D(width=1000, height=750, use_region=True)
    m3.render(
        elevation_map=elevation,
        color_map=mapcolor,
        position=(0.5, 1.0), height=1500, perspective=15,
        light_position=(0.5, 0.7, 0.8), light_brightness=70,
        fringe="ne", fringe_color="white", fringe_elevation=20,
        arrow_position=(100, 100), arrow_size=100,
    )
    m3.overlay.d_legend(
        raster=mapcolor, at=legend_at,
        font="sans", fontsize=20,
        border_color="none", title=legend_units, title_fontsize=22,
        range=f"{rmin},{rmax}", flags=legend_flags,
    )
    m3.save(filename=str(out))
    return m3

In [ ]:
with gs.RegionManager(
    raster="inundation_strds_5.0",
    res=1, res3=5, t=100, b=0, flags="ap3",
):
    # Topography + derivatives draped over the DTM
    aoi_3d(LIDAR_DTM_1M,                 legend_units="Elevation [m]").show()
    aoi_3d(f"{LIDAR_DTM_1M}_slope",      legend_units="Slope [\u00b0]").show()
    aoi_3d(f"{LIDAR_DTM_1M}_aspect",     legend_units="Aspect [\u00b0]").show()
    aoi_3d(f"{LIDAR_DTM_1M}_pcurv",      legend_units="Profile curvature").show()
    aoi_3d(f"{LIDAR_DTM_1M}_tcurv",      legend_units="Tangential curvature").show()

    # Hydrology + simulation outputs
    aoi_3d("twi",                        legend_units="TWI").show()
    aoi_3d("tpi", save_name="tpi",       legend_units="TPI").show()
    aoi_3d("max_depth",                  legend_units="Water depth [m]",
            legend_flags="bsld").show()
    aoi_3d("depth.30", elevation="depth.30",
            legend_units="Water depth [m]", legend_flags="bsld").show()

    # Solar
    aoi_3d("global_rad_172", legend_units="Global radiation [Wh/m\u00b2]").show()
    aoi_3d("global_rad_356", legend_units="Global radiation [Wh/m\u00b2]").show()

    # Flow accumulation
    aoi_3d("d8_mfd_flowaccum",
            legend_units="Flow accumulation [D8 MFD]",
            legend_range_min=1, legend_flags="blt").show()
    aoi_3d("d8_sfd_flowaccum",
            legend_units="Flow accumulation [D8 SFD]",
            legend_range_min=1, legend_flags="blt").show()
    aoi_3d("dinf_sfd_flowaccum",
            legend_units="Flow accumulation [D-infinity SFD]",
            legend_range_min=1, legend_flags="btl").show()

    # HAND + inundation
    aoi_3d("hand",
            legend_units="Height above nearest drainage [m]",
            legend_flags="bdt").show()
    aoi_3d("inundation_strds_3.0",
            save_name="inundation_strds_3.0",
            legend_units="Inundation [m]", legend_flags="bdt").show()

## Done

All chapter analyses are now available in the GRASS project at:
- `CH12/ponui/PERMANENT/` and the rendered figures are saved under
- `CH12/figures/`. To use the GRASS project from the GRASS GUI, open it with
- `grass CH12/ponui/PERMANENT`.


Please consider supporting the GRASS project by making a donation via [NumFocus](https://numfocus.org/donate-to-grass) at
[https://numfocus.org/donate-to-grass](https://numfocus.org/donate-to-grass) or join the [community](https://grass.osgeo.org/support/community/)!

### Get in Touch
- [GRASS Discord](https://discourse.osgeo.org/c/grass/62)

### GitHub Repositories
- [OSGeo/grass](https://github.com/OSGeo/grass/)
- [OSGeo/grass-addons](https://github.com/OSGeo/grass-addons/)
- [OSGeo/grass-tutorials](https://grass-tutorials.osgeo.org/)
